# Second-Stimulus Category Decoding — Group Results

Existence-proof test for GLMsingle beta contamination (session-notes 2026-09-03
finding 19): can the CATEGORY of the second (co-present) stimulus be decoded from a
beta nominally locked to first-stimulus onset? Produced by `run_stim2_decoding.py`.

4-class (face/hand/house/figure) LinearSVC, leave-one-run-out CV, chance = 0.25.

**Two feature variants:**
- `raw`: global `standardize=True` only
- `s1cat_demeaned`: per (run × stim-1-category) cell-mean subtracted — rules out
  "this is just stim-1 category patterns leaking through," since stim-1 category is
  fully confounded with run × stim-1-category cell membership.

**All trials kept** (every trial has a valid second-stimulus category, unlike the
frequency decoder). Compared against `run_decoding.py`'s stim-1 category decoding
on the same subjects/masks as a reference point for how strong the contamination is
relative to the "real" signal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests

DERIV_DIR = Path("/Users/hugofluhr/phd_local/data/LearningHabits/dev_sample/bids_dataset/derivatives")
STIM2_DIR = DERIV_DIR / "stim2_decoding"
DECODE_DIR = DERIV_DIR / "decoding"   # stim-1 category decoding, for comparison

MASKS = ["wholebrain", "visualcortex", "fusiform", "vmpfc", "striatum",
         "habit", "putamen", "premotor", "parietal"]
MASK_LABELS = {
    "wholebrain": "Whole brain", "visualcortex": "Visual cortex",
    "fusiform": "Fusiform", "vmpfc": "vmPFC", "striatum": "Striatum",
    "habit": "Habit (Guida)", "putamen": "Putamen",
    "premotor": "Premotor", "parietal": "Parietal",
}
VARIANTS = ["raw", "s1cat_demeaned"]
VARIANT_LABELS = {"raw": "Raw", "s1cat_demeaned": "Run × stim1-cat demeaned"}
CHANCE = 0.25

## 1. Data Loading & Completeness Check

In [ ]:
# Load all per-subject CSVs
records = []
cms = {mask: {variant: [] for variant in VARIANTS} for mask in MASKS}

csv_files = sorted(STIM2_DIR.glob("sub-*/sub-*_stim2_decoding.csv"))
print(f"Found {len(csv_files)} subject CSVs")

for f in csv_files:
    sid = f.parent.name.replace("sub-", "")
    sub_df = pd.read_csv(f)
    for _, row in sub_df.iterrows():
        mask = row["mask"]
        rec = {"subject": sid, "mask": mask, "n_voxels": int(row["n_voxels"]),
               "n_trials": int(row["n_trials"]), "chance": row["chance"]}
        rec["accuracy_raw"] = row["accuracy"]
        rec["accuracy_s1cat_demeaned"] = row["accuracy_s1cat_demeaned"]
        records.append(rec)

        for v in VARIANTS:
            cm_path = f.parent / f"sub-{sid}_stim2_decoding_confusion_{mask}_{v}.npy"
            if cm_path.exists():
                cms[mask][v].append(np.load(cm_path))

df = pd.DataFrame(records)
print(f"Subjects: {df['subject'].nunique()}, Masks: {df['mask'].nunique()}")
print(f"Subjects: {sorted(df['subject'].unique())}")
print(f"Chance level(s) in data: {sorted(df['chance'].unique())}")

## 2. Group Accuracy — Raw vs. Stim1-Category-Demeaned

If `raw` decoding is well above chance and `s1cat_demeaned` stays well above chance
too, that's the existence proof: the beta carries stim-2 category information that
survives removing all stim-1-category-driven variance.

In [ ]:
# Group-level stats per mask, per variant
group_stats = []
for mask in MASKS:
    mdf = df[df["mask"] == mask]
    for v in VARIANTS:
        accs = mdf[f"accuracy_{v}"].values
        t, p = stats.ttest_1samp(accs, CHANCE)
        group_stats.append({
            "mask": mask, "label": MASK_LABELS.get(mask, mask),
            "variant": v, "variant_label": VARIANT_LABELS[v],
            "mean_acc": accs.mean(), "std": accs.std(), "sem": accs.std()/np.sqrt(len(accs)),
            "t": t, "p": p, "n": len(accs),
        })

gs = pd.DataFrame(group_stats)

# FDR within the primary variant (raw) across 9 masks
primary = gs[gs["variant"] == "raw"].copy()
_, primary["p_fdr"], _, _ = multipletests(primary["p"], alpha=0.05, method="fdr_bh")
primary["sig"] = primary["p_fdr"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "")

# Same FDR pass on the demeaned variant — the one that matters for the contamination claim
demeaned = gs[gs["variant"] == "s1cat_demeaned"].copy()
_, demeaned["p_fdr"], _, _ = multipletests(demeaned["p"], alpha=0.05, method="fdr_bh")
demeaned["sig"] = demeaned["p_fdr"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "")

print("Raw variant — FDR across 9 masks:")
display(primary[["label", "mean_acc", "std", "t", "p", "p_fdr", "sig", "n"]].round(4))
print("\nStim1-cat-demeaned variant — FDR across 9 masks:")
display(demeaned[["label", "mean_acc", "std", "t", "p", "p_fdr", "sig", "n"]].round(4))

In [ ]:
# Side-by-side bar chart: raw vs s1cat-demeaned
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(MASKS))
width = 0.35

for i, v in enumerate(VARIANTS):
    means = [gs[(gs["mask"]==m) & (gs["variant"]==v)]["mean_acc"].values[0] for m in MASKS]
    sems  = [gs[(gs["mask"]==m) & (gs["variant"]==v)]["sem"].values[0] for m in MASKS]
    ax.bar(x + i*width - width/2, means, width, yerr=sems, capsize=3,
          label=VARIANT_LABELS[v], alpha=0.85)

ax.axhline(CHANCE, color="grey", ls="--", lw=1, label=f"Chance ({CHANCE})")
ax.set_xticks(x)
ax.set_xticklabels([MASK_LABELS.get(m, m) for m in MASKS], rotation=30, ha="right")
ax.set_ylabel("Mean accuracy")
ax.set_title("Second-stimulus category decoding: raw vs. stim1-cat-demeaned")
ax.legend(loc="upper right")
ax.set_ylim(0.0, max(gs["mean_acc"]) + 0.10)

for j, m in enumerate(MASKS):
    row = primary[primary["mask"] == m]
    if row["sig"].values[0]:
        mean_val = row["mean_acc"].values[0]
        sem_val = row["sem"].values[0]
        ax.text(j - width/2, mean_val + sem_val + 0.01, row["sig"].values[0],
                ha="center", va="bottom", fontsize=10)
    row_d = demeaned[demeaned["mask"] == m]
    if row_d["sig"].values[0]:
        mean_val = row_d["mean_acc"].values[0]
        sem_val = row_d["sem"].values[0]
        ax.text(j + width/2, mean_val + sem_val + 0.01, row_d["sig"].values[0],
                ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

print("\nPaired difference (raw − s1cat_demeaned) per mask:")
for m in MASKS:
    mdf = df[df["mask"] == m]
    diff = mdf["accuracy_raw"].values - mdf["accuracy_s1cat_demeaned"].values
    t, p = stats.ttest_1samp(diff, 0)
    print(f"  {MASK_LABELS.get(m,m):20s}: Δ = {diff.mean():+.4f}  t = {t:+.2f}  p = {p:.3f}")

## 3. Confusion Matrices (Group Average, s1cat-Demeaned)

Normalized by true class. Labels order comes from `sub-<id>_stim2_decoding_labels.npy`
(same across subjects — alphabetical: face, figure, hand, house).

In [ ]:
# Category labels are the same across subjects (alphabetical: face, figure, hand, house)
first_labels_file = next(STIM2_DIR.glob("sub-*/sub-*_stim2_decoding_labels.npy"), None)
cats = list(np.load(first_labels_file)) if first_labels_file else ["face", "figure", "hand", "house"]
n_cat = len(cats)
print("Categories:", cats)

def normalize_cm(cm_stack):
    normed = []
    for cm in cm_stack:
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        normed.append(cm / row_sums)
    return np.mean(normed, axis=0)

variant = "s1cat_demeaned"
n_masks = len([m for m in MASKS if cms.get(m, {}).get(variant)])
if n_masks > 0:
    fig, axes = plt.subplots(2, (n_masks + 1) // 2, figsize=(3.5 * ((n_masks+1)//2), 7))
    axes = axes.flatten()
    plot_idx = 0
    for mask in MASKS:
        stack = cms.get(mask, {}).get(variant, [])
        if not stack:
            continue
        cm_norm = normalize_cm(stack)
        ax = axes[plot_idx]
        im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
        for r in range(n_cat):
            for c in range(n_cat):
                ax.text(c, r, f"{cm_norm[r, c]:.2f}", ha="center", va="center",
                        fontsize=9, color="white" if cm_norm[r, c] > 0.5 else "black")
        ax.set_xticks(range(n_cat)); ax.set_xticklabels(cats, rotation=45, fontsize=8)
        ax.set_yticks(range(n_cat)); ax.set_yticklabels(cats, fontsize=8)
        ax.set_xlabel("Predicted (stim-2)"); ax.set_ylabel("True (stim-2)")
        ax.set_title(MASK_LABELS.get(mask, mask), fontsize=10)
        plot_idx += 1
    for i in range(plot_idx, len(axes)):
        axes[i].set_visible(False)
    fig.suptitle(f"Confusion matrices (s1cat-demeaned, n={len(stack)})", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No confusion matrices found.")

## 4. Comparison to Stim-1 Category Decoding

`run_decoding.py`'s stim-1 category decoding (whole-brain + visual cortex only,
`sub-<id>_decoding_accuracy.csv`) as a magnitude reference: how big is the stim-2
"contamination" signal relative to the intended stim-1 signal in the same betas?

In [ ]:
s1_files = sorted(DECODE_DIR.glob("sub-*/sub-*_decoding_accuracy.csv"))
if s1_files:
    s1_records = []
    for f in s1_files:
        sid = f.parent.name.replace("sub-", "")
        sub_df = pd.read_csv(f)
        for _, row in sub_df.iterrows():
            s1_records.append({"subject": sid, "mask": row["mask"], "accuracy_stim1": row["accuracy"]})
    s1_df = pd.DataFrame(s1_records)
    print(f"Stim-1 decoding: {len(s1_files)} subjects, masks = {sorted(s1_df['mask'].unique())}")

    compare_masks = [m for m in ["wholebrain", "visualcortex"] if m in s1_df["mask"].unique()]
    fig, ax = plt.subplots(figsize=(6, 5))
    x = np.arange(len(compare_masks))
    width = 0.25

    for i, (label, col, src) in enumerate([
            ("Stim-1 (raw)", "accuracy_stim1", s1_df),
            ("Stim-2 (raw)", "accuracy_raw", df),
            ("Stim-2 (s1cat-dem.)", "accuracy_s1cat_demeaned", df)]):
        means, sems = [], []
        for m in compare_masks:
            vals = src.loc[src["mask"] == m, col].values
            means.append(vals.mean()); sems.append(vals.std()/np.sqrt(len(vals)))
        ax.bar(x + (i-1)*width, means, width, yerr=sems, capsize=3, label=label, alpha=0.85)

    ax.axhline(CHANCE, color="grey", ls="--", lw=1, label=f"Chance ({CHANCE})")
    ax.set_xticks(x)
    ax.set_xticklabels([MASK_LABELS.get(m, m) for m in compare_masks])
    ax.set_ylabel("Mean accuracy")
    ax.set_title("Stim-1 vs. stim-2 category decoding, same betas")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No stim-1 decoding results found — run run_decoding.py/submit_decoding.sh first, "
          "or skip this comparison.")

## 5. Findings Summary

*To fill in once the cluster job (`submit_stim2_decoding.sh`, job 5495879) completes
and outputs are synced to `STIM2_DIR` — see §2's FDR tables and §4's stim-1 comparison.
Expected shape of the finding: raw accuracy well above chance in visual/fusiform masks
(existence proof), and if `s1cat_demeaned` accuracy stays well above chance too, that's
the decisive part — stim-2 category information survives removing all stim-1-category
structure, i.e. it isn't just stim-1 pattern leakage relabeled.*